In [ ]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "algebrax",
# ]
# [tool.uv.sources]
# algebrax = { path = ".." }
# ///

# Rubik's Cube Permutation Group, Commutators & Invariant Solver

## Theory & Mathematical Foundation

1. **The Rubik's Cube Permutation Group ($\mathcal{G} \le S_{48}$)**:
   The 3x3x3 Rubik's cube consists of 6 fixed center faces, 8 corner cubies, and 12 edge cubies,
   comprising 48 movable facelets (or 54 facelets including centers). The group $\mathcal{G}$ is
   generated by 6 clockwise 90° face turns:
   $$\mathcal{G} = \langle U, D, L, R, F, B \rangle$$
   The order of the full group is:
   $$|\mathcal{G}| = \frac{8! \cdot 3^7 \cdot 12! \cdot 2^{11}}{2} = 43{,}252{,}003{,}274{,}489{,}856{,}000$$
   $$|\mathcal{G}| \approx 4.33 \times 10^{19}$$

2. **The First Principles & Conservation Laws (Solvability Axioms)**:
   Any physically reachable state must satisfy three fundamental conservation laws:
   - **Corner Orientation Invariant**: $\sum_{i=1}^8 o(c_i) \equiv 0 \pmod 3$
   - **Edge Orientation Invariant**: $\sum_{i=1}^{12} o(e_i) \equiv 0 \pmod 2$
   - **Permutation Parity Conservation**: The product of corner and edge permutation signatures is even:
     $$\operatorname{sgn}(\sigma_{\text{corners}}) \cdot \operatorname{sgn}(\sigma_{\text{edges}}) = +1$$
     Verified directly using `algebrax.group.signature`.

3. **Commutator Subgroups and Localized Surgery**:
   The commutator $[A, B] = A B A^{-1} B^{-1}$ measures the failure of two transformations to commute.
   When the supports of $A$ and $B$ intersect in a small subset (e.g., a single cubie), $[A, B]$
   acts purely on that intersection:
   - **Corner 3-Cycle**: $[R' D' R, U] = (R' D' R) U (R' D R) U'$ cycles 3 corners with zero edge displacement.
   - **Edge 3-Cycle**: $F2 U (L R') F2 (L' R) U F2$ cycles 3 edges with zero corner displacement.
   - **Corner Twisting**: $[R' D' R D]^2 U [D' R' D R]^2 U'$ twists two corners with zero net orientation shift.

4. **Permutation Group Algebra with `algebrax.group`**:
   - Move composition: $\sigma_{AB} = \text{compose}(A, B)$
   - Inverse operations: $\sigma_{A^{-1}} = \text{invert}(A)$
   - Signature parity: $\operatorname{sgn}(\sigma) = \text{signature}(\sigma)$

In [ ]:
import random
from collections import deque
from collections.abc import Mapping
from typing import Any

import algebrax as ax

# ---------------------------------------------------------------------------
# 1. 3D Spatial Geometry & 54-Facelet Indexing
# ---------------------------------------------------------------------------

FACES = ['U', 'L', 'F', 'R', 'B', 'D']
FACE_COLORS = {'U': 'W', 'L': 'O', 'F': 'G', 'R': 'R', 'B': 'B', 'D': 'Y'}


def _rot_u(v: tuple[int, int, int]) -> tuple[int, int, int]:
    """Rotate (x, y, z) clockwise around +z axis looking from top."""
    return (-v[1], v[0], v[2])


def _rot_d(v: tuple[int, int, int]) -> tuple[int, int, int]:
    """Rotate (x, y, z) clockwise around -z axis looking from bottom."""
    return (v[1], -v[0], v[2])


def _rot_f(v: tuple[int, int, int]) -> tuple[int, int, int]:
    """Rotate (x, y, z) clockwise around +y axis looking from front."""
    return (v[2], v[1], -v[0])


def _rot_b(v: tuple[int, int, int]) -> tuple[int, int, int]:
    """Rotate (x, y, z) clockwise around -y axis looking from back."""
    return (-v[2], v[1], v[0])


def _rot_l(v: tuple[int, int, int]) -> tuple[int, int, int]:
    """Rotate (x, y, z) clockwise around -x axis looking from left."""
    return (v[0], v[2], -v[1])


def _rot_r(v: tuple[int, int, int]) -> tuple[int, int, int]:
    """Rotate (x, y, z) clockwise around +x axis looking from right."""
    return (v[0], -v[2], v[1])


_FACE_ROT_RULES = {
    'U': (lambda p: p[2] == 1, _rot_u),
    'D': (lambda p: p[2] == -1, _rot_d),
    'F': (lambda p: p[1] == 1, _rot_f),
    'B': (lambda p: p[1] == -1, _rot_b),
    'L': (lambda p: p[0] == -1, _rot_l),
    'R': (lambda p: p[0] == 1, _rot_r),
}

_FACE_LAYOUTS: dict[str, dict[str, Any]] = {
    'U': {
        'n': (0, 0, 1),
        'grid': [
            (-1, -1, 1),
            (0, -1, 1),
            (1, -1, 1),
            (-1, 0, 1),
            (0, 0, 1),
            (1, 0, 1),
            (-1, 1, 1),
            (0, 1, 1),
            (1, 1, 1),
        ],
    },
    'L': {
        'n': (-1, 0, 0),
        'grid': [
            (-1, -1, 1),
            (-1, 0, 1),
            (-1, 1, 1),
            (-1, -1, 0),
            (-1, 0, 0),
            (-1, 1, 0),
            (-1, -1, -1),
            (-1, 0, -1),
            (-1, 1, -1),
        ],
    },
    'F': {
        'n': (0, 1, 0),
        'grid': [
            (-1, 1, 1),
            (0, 1, 1),
            (1, 1, 1),
            (-1, 1, 0),
            (0, 1, 0),
            (1, 1, 0),
            (-1, 1, -1),
            (0, 1, -1),
            (1, 1, -1),
        ],
    },
    'R': {
        'n': (1, 0, 0),
        'grid': [
            (1, 1, 1),
            (1, 0, 1),
            (1, -1, 1),
            (1, 1, 0),
            (1, 0, 0),
            (1, -1, 0),
            (1, 1, -1),
            (1, 0, -1),
            (1, -1, -1),
        ],
    },
    'B': {
        'n': (0, -1, 0),
        'grid': [
            (1, -1, 1),
            (0, -1, 1),
            (-1, -1, 1),
            (1, -1, 0),
            (0, -1, 0),
            (-1, -1, 0),
            (1, -1, -1),
            (0, -1, -1),
            (-1, -1, -1),
        ],
    },
    'D': {
        'n': (0, 0, -1),
        'grid': [
            (-1, 1, -1),
            (0, 1, -1),
            (1, 1, -1),
            (-1, 0, -1),
            (0, 0, -1),
            (1, 0, -1),
            (-1, -1, -1),
            (0, -1, -1),
            (1, -1, -1),
        ],
    },
}

# Build canonical 54-facelet registry
FACELETS: list[tuple[tuple[int, int, int], tuple[int, int, int]]] = []
for _f in FACES:
    _norm = _FACE_LAYOUTS[_f]['n']
    for _pt in _FACE_LAYOUTS[_f]['grid']:
        FACELETS.append((_pt, _norm))

FACELET_INDEX: dict[tuple[tuple[int, int, int], tuple[int, int, int]], int] = {
    fl: idx for idx, fl in enumerate(FACELETS)
}

# Compute 6 primary face turn permutations in S_54
GENERATOR_PERMS: dict[str, dict[int, int]] = {}
for _name, (_filter_fn, _rot_fn) in _FACE_ROT_RULES.items():
    _perm: dict[int, int] = {}
    for _idx, (_pt, _norm) in enumerate(FACELETS):
        if _filter_fn(_pt):
            _new_pt = _rot_fn(_pt)
            _new_norm = _rot_fn(_norm)
            _perm[_idx] = FACELET_INDEX[(_new_pt, _new_norm)]
        else:
            _perm[_idx] = _idx
    GENERATOR_PERMS[_name] = _perm

# Precalculate full quarter-turn, inverse, and half-turn move dictionary
ALL_CUBE_MOVES: dict[str, dict[int, int]] = dict(GENERATOR_PERMS)
for _m in list(GENERATOR_PERMS.keys()):
    ALL_CUBE_MOVES[_m + "'"] = ax.group.invert(GENERATOR_PERMS[_m])
    ALL_CUBE_MOVES[_m + '2'] = ax.group.compose(GENERATOR_PERMS[_m], GENERATOR_PERMS[_m])

# ---------------------------------------------------------------------------
# 2. Cubie Definitions (Corners & Edges) for Invariant Audits
# ---------------------------------------------------------------------------

CubieMetadata = tuple[tuple[int, int, int], list[tuple[tuple[int, int, int], str]]]

CORNERS_METADATA: dict[str, CubieMetadata] = {
    'UFL': ((-1, 1, 1), [((0, 0, 1), 'U'), ((0, 1, 0), 'F'), ((-1, 0, 0), 'L')]),
    'UFR': ((1, 1, 1), [((0, 0, 1), 'U'), ((0, 1, 0), 'F'), ((1, 0, 0), 'R')]),
    'UBL': ((-1, -1, 1), [((0, 0, 1), 'U'), ((0, -1, 0), 'B'), ((-1, 0, 0), 'L')]),
    'UBR': ((1, -1, 1), [((0, 0, 1), 'U'), ((0, -1, 0), 'B'), ((1, 0, 0), 'R')]),
    'DFL': ((-1, 1, -1), [((0, 0, -1), 'D'), ((0, 1, 0), 'F'), ((-1, 0, 0), 'L')]),
    'DFR': ((1, 1, -1), [((0, 0, -1), 'D'), ((0, 1, 0), 'F'), ((1, 0, 0), 'R')]),
    'DBL': ((-1, -1, -1), [((0, 0, -1), 'D'), ((0, -1, 0), 'B'), ((-1, 0, 0), 'L')]),
    'DBR': ((1, -1, -1), [((0, 0, -1), 'D'), ((0, -1, 0), 'B'), ((1, 0, 0), 'R')]),
}

EDGES_METADATA: dict[str, CubieMetadata] = {
    'UB': ((0, -1, 1), [((0, 0, 1), 'U'), ((0, -1, 0), 'B')]),
    'UL': ((-1, 0, 1), [((0, 0, 1), 'U'), ((-1, 0, 0), 'L')]),
    'UR': ((1, 0, 1), [((0, 0, 1), 'U'), ((1, 0, 0), 'R')]),
    'UF': ((0, 1, 1), [((0, 0, 1), 'U'), ((0, 1, 0), 'F')]),
    'BL': ((-1, -1, 0), [((0, -1, 0), 'B'), ((-1, 0, 0), 'L')]),
    'BR': ((1, -1, 0), [((0, -1, 0), 'B'), ((1, 0, 0), 'R')]),
    'FL': ((-1, 1, 0), [((0, 1, 0), 'F'), ((-1, 0, 0), 'L')]),
    'FR': ((1, 1, 0), [((0, 1, 0), 'F'), ((1, 0, 0), 'R')]),
    'DB': ((0, -1, -1), [((0, 0, -1), 'D'), ((0, -1, 0), 'B')]),
    'DL': ((-1, 0, -1), [((0, 0, -1), 'D'), ((-1, 0, 0), 'L')]),
    'DR': ((1, 0, -1), [((0, 0, -1), 'D'), ((1, 0, 0), 'R')]),
    'DF': ((0, 1, -1), [((0, 0, -1), 'D'), ((0, 1, 0), 'F')]),
}


class RubiksCube:
    r"""Mathematical state representation of a 3x3x3 Rubik's Cube.

    Algebraic Signature:
        $\langle \mathcal{S}_{54}, \circ, \mathrm{id} \rangle$

    Carrier:
        `dict[int, str]` mapping facelet index $0 \le i < 54$ to its sticker color
        in `{'W', 'Y', 'G', 'B', 'O', 'R'}`.

    Operations:
        - `apply_move(move)`: State transition via `algebrax.group.compose`.
        - `audit_invariants()`: Popper falsifiability audit of group invariants.
    """

    def __init__(self, state: Mapping[int, str] | None = None) -> None:
        """Initialize the Rubik's cube to solved or a specified facelet state."""
        if state is None:
            self._state: dict[int, str] = {i: FACE_COLORS[FACES[i // 9]] for i in range(54)}
        else:
            self._state = dict(state)

    @property
    def state(self) -> dict[int, str]:
        """Return the current facelet state mapping."""
        return dict(self._state)

    def is_solved(self) -> bool:
        """Check if all faces have uniform color."""
        for f_idx in range(6):
            expected = FACE_COLORS[FACES[f_idx]]
            if any(self._state[f_idx * 9 + i] != expected for i in range(9)):
                return False
        return True

    def apply_move(self, move_str: str) -> 'RubiksCube':
        """Apply a move to the cube state via inverse permutation composition."""
        if move_str not in ALL_CUBE_MOVES:
            raise ValueError(f"Unknown move: '{move_str}'")
        inv_p = ax.group.invert(ALL_CUBE_MOVES[move_str])
        new_state = ax.group.compose(inv_p, self._state)
        return RubiksCube(new_state)

    def apply_sequence(self, moves: list[str] | str) -> 'RubiksCube':
        """Apply a sequence of moves to the cube."""
        move_list = moves.split() if isinstance(moves, str) else moves
        curr = self
        for m in move_list:
            curr = curr.apply_move(m)
        return curr

    def scramble(self, n_moves: int = 20, seed: int | None = None) -> tuple['RubiksCube', list[str]]:
        """Apply a pseudo-random sequence of valid face turns."""
        rng = random.Random(seed)
        all_keys = list(ALL_CUBE_MOVES.keys())
        scramble_moves = [rng.choice(all_keys) for _ in range(n_moves)]
        return self.apply_sequence(scramble_moves), scramble_moves

    def get_cubie_colors(self, cubie_meta: CubieMetadata) -> dict[str, str]:
        """Extract the current colors of a cubie by normal direction."""
        pt, faces = cubie_meta
        colors: dict[str, str] = {}
        for norm, face_char in faces:
            fid = FACELET_INDEX[(pt, norm)]
            colors[face_char] = self._state[fid]
        return colors

    def audit_invariants(self) -> dict[str, Any]:
        """Audit the fundamental conservation laws of the Rubik's Cube state.

        Returns:
            Dictionary containing corner orientation sum (mod 3), edge orientation
            sum (mod 2), corner signature, edge signature, total parity, and valid status.
        """
        # 1. Corner Orientation Sum (mod 3)
        # For each corner, distance of white/yellow sticker from U/D face
        corner_twist_sum = 0
        canonical_corners = list(CORNERS_METADATA.keys())
        corner_perm: dict[int, int] = {}

        for c_idx, c_name in enumerate(canonical_corners):
            colors = self.get_cubie_colors(CORNERS_METADATA[c_name])
            # Determine twist: 0 if U/D color on U/D, 1 if clockwise, 2 if counter-clockwise
            if ('U' in colors and colors['U'] in ('W', 'Y')) or ('D' in colors and colors['D'] in ('W', 'Y')):
                twist = 0
            elif ('F' in colors and colors['F'] in ('W', 'Y')) or ('B' in colors and colors['B'] in ('W', 'Y')):
                twist = 1
            else:
                twist = 2
            corner_twist_sum = (corner_twist_sum + twist) % 3

            # Identify piece identity by unordered set of colors
            piece_colors = set(colors.values())
            for orig_idx, orig_name in enumerate(canonical_corners):
                orig_spec = CORNERS_METADATA[orig_name][1]
                orig_colors = {FACE_COLORS[fc] for _, fc in orig_spec}
                if piece_colors == orig_colors:
                    corner_perm[orig_idx] = c_idx
                    break

        # 2. Edge Orientation Sum (mod 2)
        edge_flip_sum = 0
        canonical_edges = list(EDGES_METADATA.keys())
        edge_perm: dict[int, int] = {}

        for e_idx, e_name in enumerate(canonical_edges):
            colors = self.get_cubie_colors(EDGES_METADATA[e_name])
            is_good_edge = (
                ('U' in colors and colors['U'] in ('W', 'Y'))
                or ('D' in colors and colors['D'] in ('W', 'Y'))
                or (('F' in colors or 'B' in colors) and ('W' not in colors.values() and 'Y' not in colors.values()))
            )
            flip = 0 if is_good_edge else 1
            edge_flip_sum = (edge_flip_sum + flip) % 2

            # Identify edge piece identity
            piece_colors = set(colors.values())
            for orig_idx, orig_name in enumerate(canonical_edges):
                orig_spec = EDGES_METADATA[orig_name][1]
                orig_colors = {FACE_COLORS[fc] for _, fc in orig_spec}
                if piece_colors == orig_colors:
                    edge_perm[orig_idx] = e_idx
                    break

        sgn_corners = ax.group.signature(corner_perm)
        sgn_edges = ax.group.signature(edge_perm)
        parity = sgn_corners * sgn_edges

        is_valid = (corner_twist_sum == 0) and (edge_flip_sum == 0) and (parity == 1)

        return {
            'corner_twist_sum_mod_3': corner_twist_sum,
            'edge_flip_sum_mod_2': edge_flip_sum,
            'corner_signature': sgn_corners,
            'edge_signature': sgn_edges,
            'combined_parity': parity,
            'is_algebraically_valid': is_valid,
        }

    def render_ascii_net(self) -> str:
        """Render a 2D ASCII unfolded net of the cube."""
        lines = []

        def face_row(f_char: str, row: int) -> str:
            f_offset = FACES.index(f_char) * 9
            return ' '.join(self._state[f_offset + row * 3 + c] for c in range(3))

        # Top: U face
        for r in range(3):
            lines.append(f'      {face_row("U", r)}')
        # Middle: L, F, R, B faces
        for r in range(3):
            line = f'{face_row("L", r)} {face_row("F", r)} {face_row("R", r)} {face_row("B", r)}'
            lines.append(line)
        # Bottom: D face
        for r in range(3):
            lines.append(f'      {face_row("D", r)}')

        return '\n'.join(lines)


# ---------------------------------------------------------------------------
# 3. Commutators & Conjugates Construction
# ---------------------------------------------------------------------------


def compute_commutator(move_a: str, move_b: str) -> str:
    """Construct the commutator sequence [A, B] = A B A' B'.

    Args:
        move_a: Move sequence string for transformation A.
        move_b: Move sequence string for transformation B.

    Returns:
        The combined commutator move string.
    """
    a_inv = invert_sequence(move_a)
    b_inv = invert_sequence(move_b)
    return f'{move_a} {move_b} {a_inv} {b_inv}'


def compute_conjugate(setup_c: str, core_a: str) -> str:
    """Construct the conjugate sequence C A C^-1.

    Args:
        setup_c: Setup moves C.
        core_a: Core transformation sequence A.

    Returns:
        The conjugated move sequence.
    """
    c_inv = invert_sequence(setup_c)
    return f'{setup_c} {core_a} {c_inv}'


def invert_sequence(moves_str: str) -> str:
    """Invert a move sequence word: (M1 M2 ... Mk)^-1 = Mk^-1 ... M1^-1."""
    tokens = moves_str.split()
    inverted_tokens = []
    for token in reversed(tokens):
        if token.endswith("'"):
            inverted_tokens.append(token[:-1])
        elif token.endswith('2'):
            inverted_tokens.append(token)
        else:
            inverted_tokens.append(token + "'")
    return ' '.join(inverted_tokens)


# ---------------------------------------------------------------------------
# 4. Cayley Graph Shortest-Path Search
# ---------------------------------------------------------------------------


def solve_cayley_subgroup(
    cube: RubiksCube,
    allowed_moves: list[str] | None = None,
    max_depth: int = 5,
) -> list[str] | None:
    """Find the shortest sequence of generator moves to solved state via Cayley graph BFS.

    Args:
        cube: The scrambled Rubik's cube.
        allowed_moves: Generator subset to search over (defaults to quarter-turns).
        max_depth: Maximum exploration depth bound.

    Returns:
        Shortest move sequence list, or None if depth bound exceeded.
    """
    if cube.is_solved():
        return []

    if allowed_moves is None:
        allowed_moves = ['U', "U'", 'D', "D'", 'L', "L'", 'R', "R'", 'F', "F'", 'B', "B'"]

    # Canonical solved state tuple
    solved_tup = tuple(RubiksCube().state[i] for i in range(54))
    start_tup = tuple(cube.state[i] for i in range(54))

    queue: deque[tuple[RubiksCube, list[str]]] = deque([(cube, [])])
    visited: set[tuple[str, ...]] = {start_tup}

    while queue:
        curr_cube, path = queue.popleft()
        if len(path) >= max_depth:
            continue

        for m in allowed_moves:
            next_cube = curr_cube.apply_move(m)
            next_tup = tuple(next_cube.state[i] for i in range(54))
            if next_tup == solved_tup:
                return [*path, m]
            if next_tup not in visited:
                visited.add(next_tup)
                queue.append((next_cube, [*path, m]))

    return None


# ---------------------------------------------------------------------------
# 5. Demonstration & Auditing Pipeline
# ---------------------------------------------------------------------------


def run_demo() -> None:
    """Execute complete Rubik's group theory verification and commutator demo."""
    print('================================================================================')
    print('Step 1: Generator Orders & Permutation Signatures via algebrax.group')
    print('================================================================================')
    identity = {i: i for i in range(54)}
    for m in FACES:
        p = GENERATOR_PERMS[m]
        p2 = ax.group.compose(p, p)
        p3 = ax.group.compose(p2, p)
        p4 = ax.group.compose(p3, p)
        assert p4 == identity, f'Move {m}^4 != id'
        assert p2 != identity, f'Move {m}^2 == id'
        sig = ax.group.signature(p)
        print(f'  Generator {m}: Order=4 verified | S_54 Signature = {sig:+d}')

    print('\n================================================================================')
    print('Step 2: Commutator Isolation: Surgical 3-Cycle of Corners')
    print('================================================================================')
    cube = RubiksCube()
    # Commutator A = R' D' R, B = U -> [A, B] = R' D' R U R' D R U'
    corner_3cycle = "R' D' R U R' D R U'"
    cycled_cube = cube.apply_sequence(corner_3cycle)

    changed_corners = [
        c for c, meta in CORNERS_METADATA.items() if cycled_cube.get_cubie_colors(meta) != cube.get_cubie_colors(meta)
    ]
    changed_edges = [
        e for e, meta in EDGES_METADATA.items() if cycled_cube.get_cubie_colors(meta) != cube.get_cubie_colors(meta)
    ]

    print(f"  Applied Commutator [R' D' R, U]: '{corner_3cycle}'")
    print(f'  Displaced Corners ({len(changed_corners)}): {changed_corners}')
    print(f'  Displaced Edges   ({len(changed_edges)}): {changed_edges}')
    assert len(changed_corners) == 3
    assert len(changed_edges) == 0, 'Commutator violated edge invariance!'
    print('  -> Perfect corner 3-cycle achieved with zero edge perturbation!')

    print('\n================================================================================')
    print('Step 3: Commutator Isolation: Surgical 3-Cycle of Edges')
    print('================================================================================')
    # Pure edge 3-cycle: F2 U L R' F2 L' R U F2
    edge_3cycle = "F2 U L R' F2 L' R U F2"
    edge_cube = cube.apply_sequence(edge_3cycle)

    changed_c = [
        c for c, meta in CORNERS_METADATA.items() if edge_cube.get_cubie_colors(meta) != cube.get_cubie_colors(meta)
    ]
    changed_e = [
        e for e, meta in EDGES_METADATA.items() if edge_cube.get_cubie_colors(meta) != cube.get_cubie_colors(meta)
    ]
    print(f"  Applied Edge Commutator Sequence: '{edge_3cycle}'")
    print(f'  Displaced Corners ({len(changed_c)}): {changed_c}')
    print(f'  Displaced Edges   ({len(changed_e)}): {changed_e}')
    assert len(changed_c) == 0, 'Edge commutator violated corner invariance!'
    assert len(changed_e) == 3
    print('  -> Perfect edge 3-cycle achieved with zero corner perturbation!')

    print('\n================================================================================')
    print('Step 4: Popper Falsifiability & Conservation Laws Audit')
    print('================================================================================')
    scramble_seed = 42
    scrambled_cube, scramble_seq = cube.scramble(n_moves=15, seed=scramble_seed)
    print(f'  Scramble (15 moves): {" ".join(scramble_seq)}')
    audit = scrambled_cube.audit_invariants()
    for k, v in audit.items():
        print(f'    {k:<28}: {v}')
    assert audit['is_algebraically_valid'] is True, 'Valid scramble failed algebraic audit!'

    print('\n================================================================================')
    print("Step 5: Cayley Graph Shortest-Path Solution (God's Number on Subgroup)")
    print('================================================================================')
    # Test shortest path on a 4-move scramble
    test_scramble = ['R', 'U', "R'", "U'"]
    sub_cube = cube.apply_sequence(test_scramble)
    print(f'  Scramble: {" ".join(test_scramble)}')
    solution = solve_cayley_subgroup(sub_cube, max_depth=5)
    print(f'  Cayley BFS Optimal Solution ({len(solution or [])} moves): {" ".join(solution or [])}')
    assert solution is not None
    restored = sub_cube.apply_sequence(solution)
    assert restored.is_solved(), 'Cayley BFS solution failed to reach solved state!'
    print('  -> Restored to solved state via Cayley graph path traversal!')

    print('\n================================================================================')
    print('Step 6: Direct Group Inversion Solution')
    print('================================================================================')
    inv_solution = invert_sequence(' '.join(scramble_seq))
    restored_from_scramble = scrambled_cube.apply_sequence(inv_solution)
    assert restored_from_scramble.is_solved(), 'Inverse sequence failed to solve cube!'
    print(f'  Inverse Word: {inv_solution}')
    print('  -> Inverse permutation composition confirmed: S o S^-1 = id')


def main() -> None:
    """Entry point for CLI execution."""
    run_demo()
    print('\n================================================================================')
    print("Recipe: Rubik's Cube Permutation Group & Commutator Solver Finished!")
    print('================================================================================')


if __name__ == '__main__':
    main()